# Exploration des données de stock

Ce carnet sert à explorer les données avant de figer les traitements dans
`src/pipeline.py`. Il travaille sur le jeu d'exemple tant que l'export réel
de l'OCP n'est pas disponible ; il suffira de changer `SOURCE` ensuite.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE / "src"))

import stock
import pipeline

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

SOURCE = RACINE / "data" / "raw" / "exemple_mouvements.xlsx"
SOURCE

## 1. Chargement et premiers contrôles

In [ ]:
df = pipeline.charger(SOURCE)
print(f"{len(df):,} lignes  |  {df['code_article'].nunique()} références")
print(f"période : {df['date_mouvement'].min():%Y-%m-%d} -> {df['date_mouvement'].max():%Y-%m-%d}")
df.head()

In [ ]:
df.isna().sum()

In [ ]:
df_propre, journal = pipeline.controler(df)
journal

## 2. Consommation mensuelle

Les mois sans consommation sont remplis à zéro. C'est indispensable : sans cela
le coefficient de variation des articles à demande rare serait très sous-estimé,
et ils seraient classés X à tort.

In [ ]:
conso = pipeline.consommation_mensuelle(df_propre)
print(conso.shape)          # (références, mois)
conso.iloc[:5, :12]

In [ ]:
# Combien de mois sans aucune sortie, par référence ?
part_zeros = (conso == 0).mean(axis=1)
px.histogram(part_zeros, nbins=30,
             labels={"value": "Part des mois sans consommation"},
             title="Intermittence de la demande")

## 3. Profil et segmentation

In [ ]:
profil = pipeline.segmenter(pipeline.profil_articles(df_propre, conso))
profil.head()

In [ ]:
pd.crosstab(profil["classe_abc"], profil["classe_xyz"], margins=True)

In [ ]:
# Vérification du principe de Pareto : que pèse réellement la classe A ?
part = (profil.groupby("classe_abc")["valeur_conso_annuelle"].sum()
        / profil["valeur_conso_annuelle"].sum())
(part * 100).round(1)

## 4. Délais fournisseurs

L'écart-type du délai alimente directement la formule du stock de sécurité.
C'est souvent le levier le plus rentable : réduire `sigma_L` coûte moins cher
que d'augmenter le stock.

In [ ]:
delais = (df_propre.groupby("fournisseur")["delai_livraison"]
          .agg(["mean", "std", "count"]).round(1)
          .sort_values("std", ascending=False))
delais

## 5. Paramètres de gestion

In [ ]:
params = pipeline.calculer_parametres(profil)
params[["segment", "taux_service_cible", "stock_securite",
        "point_commande", "quantite_commande"]].head(10)

In [ ]:
# Où se trouve l'écart entre stock actuel et stock cible ?
ecart = (params.groupby("classe_abc")[["valeur_stock_actuel", "valeur_stock_cible"]]
         .sum() / 1000).round(0)
ecart["ecart"] = ecart["valeur_stock_cible"] - ecart["valeur_stock_actuel"]
ecart

## 6. Sensibilité aux hypothèses de coûts

`COUT_PASSATION` et `TAUX_POSSESSION` ne viennent pas des données : ce sont des
hypothèses. Il faut donc savoir à quel point les résultats en dépendent avant de
les présenter.

In [ ]:
lignes = []
for passation in [200, 400, 800, 1600]:
    for possession in [0.15, 0.22, 0.30]:
        q = stock.quantite_economique(params["demande_annuelle"], passation,
                                      possession * params["prix_unitaire"])
        lignes.append({"cout_passation": passation,
                       "taux_possession": possession,
                       "stock_cycle_kMAD": float((q / 2 * params["prix_unitaire"]).sum()) / 1000})

pd.DataFrame(lignes).pivot(index="cout_passation", columns="taux_possession",
                           values="stock_cycle_kMAD").round(0)

## À faire

- [ ] Remplacer `SOURCE` par l'export réel dès réception.
- [ ] Ajuster les seuils XYZ après avoir vu la vraie distribution du CV.
- [ ] Valider `COUT_PASSATION` et `TAUX_POSSESSION` avec les services concernés.
- [ ] Traiter à part les références à demande intermittente (classe Z).
